# Simulations moteur & correcteurs

### Imports

In [14]:
import numpy as np
from control import tf, c2d, feedback, damp
from control_plotly import pzmap, step, impulse, bode, nichols, rlocus

### Constantes

In [15]:
### mt_ = moteur
mt_L = 1.4e-3 #H
mt_R = 1.4 #Ohm
mt_Km = 57e-3 #Nm/A
mt_J = 2.5e-5 #Kg/m2
mt_f = 1.5e-4 #N.m/(rad/s)

s = tf('s')

Mt_s = (mt_Km)/((mt_Km**2)+(mt_f+mt_J*s)*(mt_R+mt_L*s))
print(Mt_s)


               0.057
------------------------------------
3.5e-08 s^2 + 3.521e-05 s + 0.003459



### Equations

### Simulations

In [16]:
### Simulation moteur
Mt_s_rpm = (60/(2*np.pi))*Mt_s
tm = np.linspace(start=0,stop=0.1, num = 1000)
step(Mt_s_rpm, tm, x_title="Temps (s)", y_title ="RPM")

In [17]:

Mt_s_H = Mt_s_rpm/(1+Mt_s_rpm)
step([Mt_s_H*158.33,Mt_s_rpm],t=tm, x_title = "Temps (s)", y_title ="Commande")

### Discret

In [18]:
tm_z = np.arange(0, 1, 0.01)
Mt_z_rpm = c2d(Mt_s_rpm, 0.01, 'zoh')
print(Mt_z_rpm)
step([Mt_z_rpm,Mt_s_rpm], t=tm_z)


     97.83 z + 7.318
--------------------------
z^2 - 0.3319 z + 4.276e-05

dt = 0.01



In [19]:
Mt_z_H = Mt_z_rpm/(1+Mt_z_rpm)
step(Mt_z_H, t=tm_z)

In [20]:
offset = 10**(36.664/20) #+1 for non-oscillating graph
print(offset)
nichols([Mt_z_rpm,Mt_z_rpm/offset, Mt_z_rpm/80])

68.10829367676388


In [21]:
offset_z = 80
Mt_z_Hcorr = (Mt_z_rpm/(offset_z))/(1+(Mt_z_rpm/(offset_z)))
tm_z2 = np.arange(0, 1, 0.01)
step(Mt_z_Hcorr,t=tm_z2)

In [22]:
tm_z2 = np.arange(0, 5, 0.01)
step([Mt_z_Hcorr,Mt_s_H],t=tm_z2)

### Correcteur

In [23]:
Te = 0.01

Kp = 1/(10**(36.664/20)+30)
Ti = 5*Te
Td = -30e-4
Ki = Kp*(Te/Ti)
Kd = Kp*(Td/Te)

z = tf('z')

Cz = Kp*(1+(Te/Ti)*(z/(z-1))+Td*(z-1)/(Te*z))

nichols([Cz*Mt_z_rpm,(Cz*Mt_z_rpm)/(1+Cz*Mt_z_rpm)])

In [24]:
Mt_z_corr = (Cz*Mt_z_rpm)/(1+(Cz*Mt_z_rpm))
tsim = np.arange(0,2,0.01)
step(1*Mt_z_corr*3800, tsim)

In [25]:
Mt_z_errcorr = 1/(1+(Cz*Mt_z_rpm))
step(Mt_z_errcorr*3800, tsim)

In [26]:
alpha_k = 100/(3800*(Kp+Kd+Ki))

step(1*Cz*Mt_z_errcorr*3800*alpha_k, t =np.arange(0,1,0.01), x_title="Temps (s)", y_title="Commande PWM (%)")